# W7D5_LAB_Micromanage_your_agents
- Week 7 / Day 5
- Student: Andreas Papachristophorou
- Course: AI Consulting & Integration 2026-07
- Date: 2026-08-23
---

In [1]:
%pip install -q langsmith openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from dotenv import load_dotenv
from langsmith import Client

load_dotenv()

client = Client()

print("LangSmith endpoint:", os.getenv("LANGSMITH_ENDPOINT"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))
print("LangSmith key found:", bool(os.getenv("LANGSMITH_API_KEY")))
print("OpenAI key found:", bool(os.getenv("OPENAI_API_KEY")))

LangSmith endpoint: https://eu.api.smith.langchain.com
LangSmith project: eu-ets-fueleu-evaluation
LangSmith key found: True
OpenAI key found: True


In [3]:
# Validate the JSON
import json

with open("eu_ets_fueleu_examples.json", "r", encoding="utf-8") as file:
    examples = json.load(file) # opens the JSON file and converts it into a Python list.

required_keys = {"inputs", "outputs", "metadata"}

print("Number of examples:", len(examples)) # counts dataset rows; it should be 12.
print("All examples have required keys:", all(
    required_keys == set(example.keys()) # checks that every example has exactly inputs, outputs, and metadata.
    for example in examples
))

print("Regulations:", sorted({
    example["metadata"]["regulation"]
    for example in examples
}))

Number of examples: 20
All examples have required keys: True
Regulations: ['EU ETS', 'EU ETS and FuelEU Maritime', 'FuelEU Maritime']


In [4]:
# Import the LangSmith Client class so Python can communicate with LangSmith.
from langsmith import Client

# Create a client using your LangSmith API key from the environment variables.
client = Client()

# Choose a clear, reusable name for your dataset in LangSmith.
DATASET_NAME = "eu-ets-fueleu-maritime-qa"

# Add a short explanation so the dataset is understandable in the LangSmith UI.
DATASET_DESCRIPTION = (
    "Question-answer evaluation dataset covering EU ETS maritime "
    "and FuelEU Maritime requirements."
)

# Get the names of datasets that already exist in your LangSmith workspace.
# This prevents accidentally uploading the same 12 examples twice.
existing_names = {dataset.name for dataset in client.list_datasets()}

# Only create and upload the dataset if that name does not already exist.
if DATASET_NAME in existing_names:
    # Tell us that the dataset already exists instead of creating a duplicate.
    print(f"Dataset already exists: {DATASET_NAME}")

else:
    # Create the empty dataset folder in LangSmith and save its details in 'dataset'.
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description=DATASET_DESCRIPTION
    )

    # Upload all examples at once.
    # The list comprehensions pull the matching inputs, expected answers,
    # and metadata from every example in the 'examples' list.
    client.create_examples(
        inputs=[example["inputs"] for example in examples],
        outputs=[example["outputs"] for example in examples],
        metadata=[example["metadata"] for example in examples],
        dataset_id=dataset.id  # Connect the uploaded examples to the new dataset.
    )

    # Confirm that the upload completed and display the number uploaded.
    print(f"Created dataset and uploaded {len(examples)} examples.")

Dataset already exists: eu-ets-fueleu-maritime-qa


In [5]:
# Import the OpenAI client and LangSmith tools for tracing.
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

# Wrap the OpenAI client so its model calls are automatically traced in LangSmith.
openai_client = wrap_openai(OpenAI())

# Label this function in LangSmith and record each time it runs.
@traceable(name="maritime-regulation-qa")
def answer_maritime_question(inputs: dict) -> dict:
    # Get the source information and question from one dataset example.
    context = inputs["context"]
    question = inputs["question"]

    # Ask the LLM to answer using only the supplied context.
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer maritime-regulation questions using only the "
                    "provided context. Give a concise, factual answer. "
                    "If the context does not contain the answer, say "
                    "'The provided context does not contain the answer.'"
                )
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {question}"
            }
        ]
    )

    # Return a dictionary whose 'answer' key matches the dataset output format.
    return {"answer": response.choices[0].message.content.strip()}

In [6]:
# Give the first dataset example to the target function.
test_result = answer_maritime_question(examples[0]["inputs"])

# Show the dictionary returned by the model.
print(test_result)

{'answer': 'Cargo and passenger ships of 5,000 gross tonnage or above have been covered by the EU ETS since 2024.'}


In [7]:
# Install or update OpenEvals in the same Python environment used by this notebook.
%pip install -U openevals
%pip install -U ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Import OpenEvals tools for creating an LLM-as-a-judge evaluator.
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

# Create a judge that compares a model answer with the dataset reference answer.
# The feedback key "correctness" is the column name used in LangSmith results.
judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-4o-mini",
    feedback_key="correctness"
)

# Create the evaluator function LangSmith will call for each example.
def correctness_evaluator(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> dict:
    # Send the original question/context, model answer, and expected answer to the judge.
    return judge(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs
    )

In [9]:
# Confirm that OpenEvals is now installed and importable.
import openevals

print("OpenEvals imported successfully.")

OpenEvals imported successfully.


In [10]:
# Run the target function on every example in the LangSmith dataset.
# The target function must be the first positional argument.
results = client.evaluate(
    answer_maritime_question,                 # The model-answering function to test.
    data=DATASET_NAME,                        # The LangSmith dataset to use.
    evaluators=[correctness_evaluator],       # The grader function to apply to each answer.
    experiment_prefix="gpt-4o-mini-maritime", # Clear label for this experiment run.
    max_concurrency=2                         # Run only two examples at a time.
)

# Display the experiment-results object after the run completes.
print(results)

View the evaluation results for experiment: 'gpt-4o-mini-maritime-492cdd55' at:
https://eu.smith.langchain.com/o/47abe72e-ff1c-4427-b640-c999e9690abc/datasets/3a62251c-045d-4040-872e-108e8736ccd8/compare?selectedSessions=3fbd762f-67a2-4307-a7ec-d609994aa4b1




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-mini-maritime-492cdd55>


In [11]:
# Read the existing dataset from LangSmith using its saved name.
dataset = client.read_dataset(dataset_name=DATASET_NAME)

# Count how many examples are already stored online.
existing_examples = list(
    client.list_examples(dataset_id=dataset.id)
)

# Keep only examples after the first 12 in the local JSON list.
# Python slice [12:] means "start at item 13 and continue to the end."
new_examples = examples[12:]

# Upload the eight new examples only if the online dataset still has 12 rows.
if len(existing_examples) == 12:
    client.create_examples(
        inputs=[example["inputs"] for example in new_examples],
        outputs=[example["outputs"] for example in new_examples],
        metadata=[example["metadata"] for example in new_examples],
        dataset_id=dataset.id
    )

    print(f"Uploaded {len(new_examples)} new examples.")

else:
    print(
        f"Upload skipped: LangSmith already has "
        f"{len(existing_examples)} examples."
    )

Upload skipped: LangSmith already has 20 examples.


In [12]:
# Evaluate all 20 examples, including the new hard and insufficient-context cases.
results_20 = client.evaluate(
    answer_maritime_question,                    # The function being tested.
    data=DATASET_NAME,                           # The updated 20-example LangSmith dataset.
    evaluators=[correctness_evaluator],          # Score each answer for correctness.
    experiment_prefix="gpt-4o-mini-maritime-20", # Identifies this expanded evaluation.
    max_concurrency=2                            # Process two cases at once.
)

# Show the experiment object and LangSmith results link.
print(results_20)

View the evaluation results for experiment: 'gpt-4o-mini-maritime-20-8af37bfb' at:
https://eu.smith.langchain.com/o/47abe72e-ff1c-4427-b640-c999e9690abc/datasets/3a62251c-045d-4040-872e-108e8736ccd8/compare?selectedSessions=d6fe0481-2ba1-4b9f-be22-3e80c26d7022




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-mini-maritime-20-8af37bfb>


In [14]:
# Wait for LangSmith evaluation work to finish
results.wait()

# Export this 20-question experiment into a pandas DataFrame
df_results = results.to_pandas()

# Check the table and its columns
print(df_results.shape)
print(df_results.columns.tolist())

(20, 9)
['inputs.context', 'inputs.question', 'outputs.answer', 'error', 'reference.answer', 'feedback.correctness', 'execution_time', 'example_id', 'id']


In [15]:
analysis_df = df_results[[
    "inputs.question",
    "reference.answer",
    "outputs.answer",
    "feedback.correctness",
    "execution_time"
]].copy()

analysis_df.columns = [
    "question",
    "reference_answer",
    "model_answer",
    "correctness",
    "execution_time_seconds"
]

analysis_df["is_correct"] = analysis_df["correctness"] == 1.0

analysis_df.head()

,question,reference_answer,model_answer,correctness,execution_time_seconds,is_correct
0,A fuel has low carbon dioxide emissions when b...,Yes. A well-to-wake assessment covers carbon d...,"Yes, a well-to-wake FuelEU assessment would co...",True,1.057781,True
1,A company is preparing its 2026 EU ETS surrend...,It must cover 70% of its emissions reported fo...,The company must cover 70% of its emissions re...,True,0.906260,True
2,"Does FuelEU Maritime apply to a 6,000-GT ship ...","Yes. FuelEU Maritime applies to ships above 5,...","Yes, FuelEU Maritime applies to a 6,000-GT shi...",True,0.949660,True
3,If a company borrows 100 units of compliance s...,110 units are deducted from the next reporting...,110 units are deducted from the next reporting...,True,0.737665,True
4,Which companies offer verification services fo...,The provided context does not contain the answer.,The provided context does not contain the answer.,True,1.979610,True


In [18]:
import pandas as pd

summary = pd.DataFrame({
    "metric": [
        "Total questions",
        "Correct answers",
        "Incorrect answers",
        "Accuracy",
        "Mean correctness score",
        "Median correctness score",
        "Average execution time (seconds)",
        "Median execution time (seconds)"
    ],
    "value": [
        len(analysis_df),
        analysis_df["is_correct"].sum(),
        (~analysis_df["is_correct"]).sum(),
        analysis_df["is_correct"].mean(),
        analysis_df["correctness"].mean(),
        analysis_df["correctness"].median(),
        analysis_df["execution_time_seconds"].mean(),
        analysis_df["execution_time_seconds"].median()
    ]
})

summary

,metric,value
0,Total questions,20.000000
1,Correct answers,19.000000
2,Incorrect answers,1.000000
3,Accuracy,0.950000
4,Mean correctness score,0.950000
5,Median correctness score,1.000000
6,Average execution time (seconds),0.814666
7,Median execution time (seconds),0.711674


In [19]:
errors = analysis_df[analysis_df["is_correct"] == False]

errors[[
    "question",
    "reference_answer",
    "model_answer",
    "correctness",
    "execution_time_seconds"
]]

,question,reference_answer,model_answer,correctness,execution_time_seconds
6,"In 2031, must a container ship use on-shore po...",Not necessarily. The requirement from 1 Januar...,The provided context does not contain the answer.,False,0.486865
